In [1]:
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
import string
from nltk.stem import WordNetLemmatizer

In [2]:
data=pd.read_csv('../input/nlp-getting-started/train.csv',index_col=0)
test=pd.read_csv('../input/nlp-getting-started/test.csv',index_col=0)

In [3]:
data.fillna('',inplace=True)

In [4]:
test.fillna('',inplace=True)

In [5]:
data.head()

,keyword,location,text,target
id,,,,
1,,,Our Deeds are the Reason of this #earthquake M...,1
4,,,Forest fire near La Ronge Sask. Canada,1
5,,,All residents asked to 'shelter in place' are ...,1
6,,,"13,000 people receive #wildfires evacuation or...",1
7,,,Just got sent this photo from Ruby #Alaska as ...,1


In [6]:
test.head()

,keyword,location,text
id,,,
0,,,Just happened a terrible car crash
2,,,"Heard about #earthquake is different cities, s..."
3,,,"there is a forest fire at spot pond, geese are..."
9,,,Apocalypse lighting. #Spokane #wildfires
11,,,Typhoon Soudelor kills 28 in China and Taiwan


In [7]:
finaldata=data['keyword']+' '+data['location']+' '+data['text']
testdata=test['keyword']+' '+test['location']+' '+test['text']
finaltarget=data['target']

In [8]:
finaldata.head()

id
1      Our Deeds are the Reason of this #earthquake...
4               Forest fire near La Ronge Sask. Canada
5      All residents asked to 'shelter in place' ar...
6      13,000 people receive #wildfires evacuation ...
7      Just got sent this photo from Ruby #Alaska a...
dtype: object

In [9]:
testdata.head()

id
0                    Just happened a terrible car crash
2       Heard about #earthquake is different cities,...
3       there is a forest fire at spot pond, geese a...
9              Apocalypse lighting. #Spokane #wildfires
11        Typhoon Soudelor kills 28 in China and Taiwan
dtype: object

In [10]:
finaltarget.head()

id
1    1
4    1
5    1
6    1
7    1
Name: target, dtype: int64

In [11]:
WNL=WordNetLemmatizer()
def text_process(data):
    msg=[c for c in data if c not in string.punctuation]
    msg=''.join(msg)
    msg=[word for word in msg.split() if word.lower() not in stopwords.words('english')]
    msg=[WNL.lemmatize(word) for word in msg ]
    return msg

In [12]:
finaldata.head(10).apply(text_process)

id
1     [Deeds, Reason, earthquake, May, ALLAH, Forgiv...
4         [Forest, fire, near, La, Ronge, Sask, Canada]
5     [resident, asked, shelter, place, notified, of...
6     [13000, people, receive, wildfire, evacuation,...
7     [got, sent, photo, Ruby, Alaska, smoke, wildfi...
8     [RockyFire, Update, California, Hwy, 20, close...
10    [flood, disaster, Heavy, rain, cause, flash, f...
13                     [Im, top, hill, see, fire, wood]
14    [Theres, emergency, evacuation, happening, bui...
15                  [Im, afraid, tornado, coming, area]
dtype: object

In [13]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfTransformer

In [14]:
CV=CountVectorizer(analyzer=text_process)

In [15]:
CV.fit(finaldata)

CountVectorizer(analyzer=<function text_process at 0x7f1682e58cb0>)

In [16]:
CV.vocabulary_

{'Deeds': 3729,
 'Reason': 9947,
 'earthquake': 15994,
 'May': 7872,
 'ALLAH': 954,
 'Forgive': 4962,
 'u': 27006,
 'Forest': 4958,
 'fire': 16539,
 'near': 23693,
 'La': 7107,
 'Ronge': 10253,
 'Sask': 10646,
 'Canada': 2746,
 'resident': 25097,
 'asked': 13786,
 'shelter': 25626,
 'place': 24374,
 'notified': 23840,
 'officer': 23925,
 'evacuation': 16239,
 'order': 24031,
 'expected': 16311,
 '13000': 159,
 'people': 24249,
 'receive': 24926,
 'wildfire': 27506,
 'California': 2709,
 'got': 16967,
 'sent': 25543,
 'photo': 24312,
 'Ruby': 10302,
 'Alaska': 1239,
 'smoke': 25852,
 'pours': 24512,
 'school': 25436,
 'RockyFire': 10218,
 'Update': 12401,
 'Hwy': 6080,
 '20': 292,
 'closed': 14901,
 'direction': 15699,
 'due': 15941,
 'Lake': 7126,
 'County': 3321,
 'CAfire': 2509,
 'flood': 16601,
 'disaster': 15715,
 'Heavy': 5798,
 'rain': 24818,
 'cause': 14675,
 'flash': 16581,
 'flooding': 16603,
 'street': 26195,
 'Manitou': 7767,
 'Colorado': 3164,
 'Springs': 11216,
 'area': 13

In [17]:
finaldata=CV.transform(finaldata)

In [18]:
finaldata.nnz

88033

In [19]:
Tfidf=TfidfTransformer()
Tfidf.fit(finaldata)
Tfidf_val=Tfidf.transform(finaldata)

In [20]:
testdata=CV.transform(testdata)
testdata=Tfidf.transform(testdata)

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

# Naive Bayes & Random Forest

In [22]:
sentiment=MultinomialNB()
sentiment.fit(Tfidf_val,finaltarget)
result1=sentiment.predict(testdata)

RFC=RandomForestClassifier()
RFC.fit(Tfidf_val,finaltarget)
result2=RFC.predict(testdata)

In [23]:
rslt=pd.DataFrame(index=test.index)
rslt['Naive Bayes']=result1
rslt['Random Forest']=result2

In [24]:
rslt['Naive Bayes'].value_counts()

0    2215
1    1048
Name: Naive Bayes, dtype: int64

In [25]:
rslt['Random Forest'].value_counts()

0    2138
1    1125
Name: Random Forest, dtype: int64

# Support Vector Machine

In [26]:
from sklearn.svm import SVC

In [27]:
svc=SVC()

In [28]:
svc.fit(Tfidf_val,finaltarget)
svm=svc.predict(testdata)

In [29]:
rslt['svm']=svm

In [30]:
rslt['svm'].value_counts()

0    2163
1    1100
Name: svm, dtype: int64

### With Grid Search

In [31]:
from sklearn.model_selection import GridSearchCV
grid={'C':[0.01,0.1,1,10,100],'gamma':[0.001,.01,0.1,1,10]}

In [32]:
g=GridSearchCV(SVC(),grid,verbose=2)

In [33]:
g.fit(Tfidf_val,finaltarget)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV] C=0.01, gamma=0.001 .............................................


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV] .............................. C=0.01, gamma=0.001, total=   6.2s
[CV] C=0.01, gamma=0.001 .............................................


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    6.2s remaining:    0.0s


[CV] .............................. C=0.01, gamma=0.001, total=   6.2s
[CV] C=0.01, gamma=0.001 .............................................
[CV] .............................. C=0.01, gamma=0.001, total=   6.2s
[CV] C=0.01, gamma=0.001 .............................................
[CV] .............................. C=0.01, gamma=0.001, total=   6.2s
[CV] C=0.01, gamma=0.001 .............................................
[CV] .............................. C=0.01, gamma=0.001, total=   6.3s
[CV] C=0.01, gamma=0.01 ..............................................
[CV] ............................... C=0.01, gamma=0.01, total=   6.2s
[CV] C=0.01, gamma=0.01 ..............................................
[CV] ............................... C=0.01, gamma=0.01, total=   6.3s
[CV] C=0.01, gamma=0.01 ..............................................
[CV] ............................... C=0.01, gamma=0.01, total=   6.2s
[CV] C=0.01, gamma=0.01 ..............................................
[CV] .

[Parallel(n_jobs=1)]: Done 125 out of 125 | elapsed: 13.9min finished


GridSearchCV(estimator=SVC(),
             param_grid={'C': [0.01, 0.1, 1, 10, 100],
                         'gamma': [0.001, 0.01, 0.1, 1, 10]},
             verbose=2)

In [34]:
g.best_estimator_

SVC(C=10, gamma=1)

In [35]:
grid=g.predict(testdata)

In [36]:
rslt['grid']=grid

In [37]:
rslt['grid'].value_counts()

0    2061
1    1202
Name: grid, dtype: int64

In [38]:
rslt.head()

,Naive Bayes,Random Forest,svm,grid
id,,,,
0,1,1,1,1
2,0,1,1,1
3,1,1,1,1
9,1,1,1,1
11,1,1,1,1


In [39]:
#rslt.to_csv('prediction.csv')